## U24AI038 NLP ASSIGNMENT 6

In [1]:
from pathlib import Path
from collections import defaultdict
data = Path("../assignment 1/tokenized_words.txt")
unigrams = defaultdict(int)
bigrams = defaultdict(int)
trigrams = defaultdict(int)
quadgrams = defaultdict(int)
dev = []
test = []
with data.open('r', encoding ='utf-8') as f:
    for i, line in enumerate(f):
        if i <10_00_000:
            if len(dev) <1000:
                dev.append(line)
            elif len(test) <1000:
                test.append(line)
            uni = line.split()
            bi = ["<s>"] + uni + ["</s>"]
            tri = ["<s>","<s>"] + uni + ["</s>"]
            quad = ["<s>", "<s>", "<s>"] + uni + ["</s>"]

            for word in uni:
                unigrams[word]+=1

            bi = zip(bi, bi[1:])
            for bigram in bi:
                bigrams[bigram] +=1

            tri = zip(tri, tri[1:], tri[2:])
            for trigram in tri:
                trigrams[trigram]+=1

            quad = zip(quad, quad[1:], quad[2:], quad[3:])
            for quadgram in quad:
                quadgrams[quadgram] +=1
        else:
            break



print(f"Unigram Counts: {len(unigrams)}")
print(f"Bigram Counts: {len(bigrams)}")
print(f"Trigram Counts: {len(trigrams)}")
print(f"Quadgram Counts: {len(quadgrams)}")




Unigram Counts: 755566
Bigram Counts: 6004138
Trigram Counts: 10834281
Quadgram Counts: 13060259


In [2]:
import math
def unigram_model(sentence):
    curr = sentence.split()
    n = sum(unigrams.values())
    ans = 0
    for word in curr:
        if unigrams[word] == 0:
            return -math.inf
        ans += math.log(unigrams[word]/n)

    return ans 

def bigram_model(sentence):
    curr = sentence.split()
    curr = ["<s>"] + curr + ["</s>"]
    ans =0
    for i in range(1, len(curr)):
        if curr[i-1] == "<s>":
            context_count = 10_00_000
        else:
            context_count = unigrams[curr[i-1]]

        curr_bigram = (curr[i-1], curr[i])
        if bigrams.get(curr_bigram,0) == 0:
            return -math.inf
        ans += math.log(bigrams[curr_bigram]/context_count)
    return ans

def trigram_model(sentence):
    curr = sentence.split()
    curr = ["<s>","<s>"] + curr + ["</s>"]
    ans =0
    for i in range(2, len(curr)):
        wi = curr[i]
        wi_1 = curr[i-1]
        wi_2 = curr[i-2]
        context = bigrams.get((wi_2,wi_1),0)
        if context == 0:
            return -math.inf
        curr_count = trigrams.get((wi_2,wi_1, wi),0)
        if curr_count ==0:
            return -math.inf

        ans += math.log(curr_count/context)
    return ans

def quadgram_model(sentence):
    curr = sentence.split()
    curr = ["<s>","<s>","<s>"] + curr + ["</s>"]
    ans =0
    for i in range(3, len(curr)):
        wi = curr[i]
        wi_1 = curr[i-1]
        wi_2 = curr[i-2]
        wi_3 = curr[i-3]
        context = trigrams.get((wi_3,wi_2,wi_1),0)
        if context == 0:
            return -math.inf
        curr_count = quadgrams.get((wi_3, wi_2,wi_1, wi),0)
        if curr_count ==0:
            return -math.inf

        ans += math.log(curr_count/context)
    return ans
    
        

#### Interpolated Smoothing (For Bigram, Trigram, and Quadgram Models)
p(wi|wi-1,wi-2,wi-3) = a1*p(wi|wi-1,wi-2,wi-3) + a2*p(wi|wi-1, wi-2) + a3*p(wi|wi-1) + a4*p(wi), a1+a2+a3+a4 =1

In [3]:
#taking constant weights: a1 = 0.4, a2 =0.3, a3 = 0.2, a4 =0.1
a1 = 0.4
a2 =0.3
a3 = 0.2
a4 =0.1
n = sum(unigrams.values())
def interpolated_model(sentence):
    ans = 0
    words = ["<s>","<s>","<s>"] + sentence.split() + ["</s>"]
    for i in range(3, len(words)):
        wi = words[i]
        wi_1 = words[i-1]
        wi_2 = words[i-2]
        wi_3 = words[i-3]
        curr_quadgrams = quadgrams.get((wi_3, wi_2, wi_1, wi),0)
        curr_trigrams = trigrams.get((wi_2,wi_1, wi), 0)
        curr_bigrams = bigrams.get((wi_1,wi), 0)
        curr_unigrams = unigrams.get((wi), 0)

        quad_factor = 0 if curr_trigrams ==0 else a1* curr_quadgrams/curr_trigrams
        tri_factor = 0 if curr_bigrams ==0 else a2* curr_trigrams/curr_bigrams
        bi_factor = 0 if curr_unigrams ==0 else a3* curr_bigrams/curr_unigrams
        uni_factor = 0 if n ==0 else a4* curr_unigrams/n

        total = uni_factor + bi_factor + tri_factor + quad_factor
        if total == 0:
            return -math.inf
        ans += math.log(total)

    return ans



    


#### testing model

In [4]:
interpolated_log_prob = 0
test_tokens =0
for sentence in test:
    test_tokens += len(sentence.split())
    interpolated_log_prob += interpolated_model(sentence)

print(f"Average log probability for interpolated model: {interpolated_log_prob/test_tokens}")
pp_interpolated = math.exp(-interpolated_log_prob/test_tokens)
print("perplexity: ", pp_interpolated)

Average log probability for interpolated model: -1.0951918333070185
perplexity:  2.989756163197306


#### 2. Good Turing Smoothing

In [5]:
freq_table_uni = defaultdict(int)
freq_table_bi = defaultdict(int)
freq_table_tri = defaultdict(int)
freq_table_quad = defaultdict(int)

for key in unigrams:
    val = unigrams[key]
    freq_table_uni[val] +=1

for key in bigrams:
    val = bigrams[key]
    freq_table_bi[val] +=1

for key in trigrams:
    val = trigrams[key]
    freq_table_tri[val] +=1

for key in quadgrams:
    val = quadgrams[key]
    freq_table_quad[val] +=1

def gt_model_uni(sentence):
    tokens = sentence.split()
    ans =0
    for token in tokens:
        curr_count = unigrams.get(token,0)
        if curr_count ==0:
            adjusted_count = freq_table_uni[1]
        else:
            if freq_table_uni[curr_count] == 0 or freq_table_uni[curr_count+1] ==0:
                return -math.inf
            adjusted_count = (curr_count+1)* (freq_table_uni[curr_count+1]/ freq_table_uni[curr_count])

        ans += math.log(adjusted_count/n)

    return ans

for sentence in test:
    val =0
    val += gt_model_uni(sentence)

print(val)

    


    

-inf


### The values of Nc can be missing and therefore we need to use estimate smoothed Nc

In [6]:
import math
import numpy as np

def smooth_frequency_table(freq_table):
    points = []

    for c, Nc in freq_table.items():
        if c > 0 and Nc > 0:
            points.append((math.log(c), math.log(Nc)))

    x = np.array([p[0] for p in points])
    y = np.array([p[1] for p in points])

    b, a = np.polyfit(x, y, 1)

    def estimated_Nc(c):
        if c <= 0:
            return 0.0
        return math.exp(a + b * math.log(c))

    return estimated_Nc

In [7]:
Nc_uni = smooth_frequency_table(freq_table_uni)
Nc_bi = smooth_frequency_table(freq_table_bi)
Nc_tri = smooth_frequency_table(freq_table_tri)
Nc_quad = smooth_frequency_table(freq_table_quad)

In [8]:
def gt_model_uni(sentence):
    tokens = sentence.split()
    ans = 0.0

    for token in tokens:
        c = unigrams.get(token, 0)

        if c == 0:
            adjusted_count = Nc_uni(1)
        else:
            adjusted_count = (c + 1) * Nc_uni(c + 1) / Nc_uni(c)

        probability = adjusted_count / n

        if probability <= 0:
            return -math.inf

        ans += math.log(probability)

    return ans
print(gt_model_uni(test[0]))


-1270.910299203417


In [9]:
def gt_model_bi(sentence):
    words = ["<s>"] + sentence.split() + ["</s>"]
    ans = 0.0

    for i in range(1, len(words)):
        w1 = words[i - 1]
        w2 = words[i]

        c = bigrams.get((w1, w2), 0)

        if w1 == "<s>":
            context_count = 1_000_000
        else:
            context_count = unigrams.get(w1, 0)

        if context_count == 0:
            return -math.inf

        if c == 0:
            adjusted_count = Nc_bi(1)
        else:
            adjusted_count = (c + 1) * Nc_bi(c + 1) / Nc_bi(c)

        probability = adjusted_count / context_count

        if probability <= 0:
            return -math.inf

        ans += math.log(probability)

    return ans

def gt_model_tri(sentence):
    words = ["<s>", "<s>"] + sentence.split() + ["</s>"]
    ans = 0.0

    for i in range(2, len(words)):
        w1 = words[i - 2]
        w2 = words[i - 1]
        w3 = words[i]

        c = trigrams.get((w1, w2, w3), 0)
        if w2 =="<s>" and w1 =="<s>":
            context_count = 10_00_000
        else:
            context_count = bigrams.get((w1, w2), 0)

        if context_count == 0:
            return -math.inf

        if c == 0:
            adjusted_count = Nc_tri(1)
        else:
            adjusted_count = (c + 1) * Nc_tri(c + 1) / Nc_tri(c)

        probability = adjusted_count / context_count

        if probability <= 0:
            return -math.inf

        ans += math.log(probability)

    return ans

def gt_model_quad(sentence):
    words = ["<s>", "<s>", "<s>"] + sentence.split() + ["</s>"]
    ans = 0.0

    for i in range(3, len(words)):
        w1 = words[i - 3]
        w2 = words[i - 2]
        w3 = words[i - 1]
        w4 = words[i]

        c = quadgrams.get((w1, w2, w3, w4), 0)
        if w1 == "<s>" and w2 == "<s>" and w3 =="<s>":
            context_count = 10_00_000
        else:
            context_count = trigrams.get((w1, w2, w3), 0)
        
        if context_count == 0:
            return -math.inf

        if c == 0:
            adjusted_count = Nc_quad(1)
        else:
            adjusted_count = (c + 1) * Nc_quad(c + 1) / Nc_quad(c)

        probability = adjusted_count / context_count

        if probability <= 0:
            return -math.inf

        ans += math.log(probability)

    return ans

In [10]:
gt_avg_log_prob_uni =0
gt_avg_log_prob_bi =0
gt_avg_log_prob_tri =0
gt_avg_log_prob_quad =0
total_tokens =0
for sentence in test:
    total_tokens += len(sentence.split())
    gt_avg_log_prob_uni += gt_model_uni(sentence)
    gt_avg_log_prob_bi += gt_model_bi(sentence)
    gt_avg_log_prob_tri += gt_model_tri(sentence)
    gt_avg_log_prob_quad += gt_model_quad(sentence)

gt_avg_log_prob_uni /= total_tokens
gt_avg_log_prob_bi /= total_tokens
gt_avg_log_prob_tri /= total_tokens
gt_avg_log_prob_quad /= total_tokens

print(f"Average log_prob of uni: {gt_avg_log_prob_uni}")
print(f"Average log_prob of bi: {gt_avg_log_prob_bi}")
print(f"Average log_prob of tri: {gt_avg_log_prob_tri}")
print(f"Average log_prob of quad: {gt_avg_log_prob_quad}")




Average log_prob of uni: -8.921881151676805
Average log_prob of bi: -4.744162377696439
Average log_prob of tri: -2.1561590046511387
Average log_prob of quad: -1.2862288405935147


In [11]:
pp_gt_uni = math.exp(-gt_avg_log_prob_uni)
pp_gt_bi = math.exp(-gt_avg_log_prob_bi)
pp_gt_tri = math.exp(-gt_avg_log_prob_tri)
pp_gt_quad = math.exp(-gt_avg_log_prob_quad)

print("GT Unigram perplexity:", pp_gt_uni)
print("GT Bigram perplexity:", pp_gt_bi)
print("GT Trigram perplexity:", pp_gt_tri)
print("GT Quadgram perplexity:", pp_gt_quad)

GT Unigram perplexity: 7494.173655401847
GT Bigram perplexity: 114.91151273425179
GT Trigram perplexity: 8.637895740942092
GT Quadgram perplexity: 3.619112538216836


#### Katz Backoff
>- If C(wi|wi-1,wi-2, wi-3) >0, pkatz = good turing as we use that as discount.
>- If count is zero, backoff to lower model multiplied by alpha
>- base case: for unigram model, its good turing and we stop there.


In [12]:
import math



boundary_context = {
    ("<s>",),
    ("<s>", "<s>"),
    ("<s>", "<s>", "<s>")
}

sentence_count = 10_00_000



def katz_probability(wi, context):

    # UNIGRAM


    if len(context) == 0:

        c = unigrams.get(wi, 0)

        if c == 0:

            adjusted_count = Nc_uni(1)

        else:

            adjusted_count = (
                (c + 1)
                * Nc_uni(c + 1)
                / Nc_uni(c)
            )

        return math.log(adjusted_count / n)




    elif len(context) == 3:

        # Convert reverse context to chronological n-gram
        #
        # context = [wi-1, wi-2, wi-3]
        #
        # => (wi-3, wi-2, wi-1, wi)

        curr_gram = tuple(
            reversed([wi] + context)
        )


        c = quadgrams.get(curr_gram, 0)



        if c == 0:

            log_alpha = alpha_quad[
                (context[0], context[1], context[2])
            ]

            return (
                katz_probability(
                    wi,
                    context[:-1]
                )
                + log_alpha
            )


        adjusted_count = (
            (c + 1)
            * Nc_quad(c + 1)
            / Nc_quad(c)
        )


        # Chronological history:
        #
        # (wi-3, wi-2, wi-1)

        curr_context = tuple(
            reversed(context)
        )


        if curr_context in boundary_context:

            context_count = sentence_count

        else:

            context_count = trigrams[curr_context]


        return math.log(
            adjusted_count / context_count
        )


    elif len(context) == 2:

        # context = [wi-1, wi-2]
        #
        # chronological trigram:
        # (wi-2, wi-1, wi)

        curr_gram = tuple(
            reversed([wi] + context)
        )


        c = trigrams.get(curr_gram, 0)


        if c == 0:

            log_alpha = alpha_tri[
                (context[0], context[1])
            ]

            return (
                katz_probability(
                    wi,
                    context[:-1]
                )
                + log_alpha
            )




        adjusted_count = (
            (c + 1)
            * Nc_tri(c + 1)
            / Nc_tri(c)
        )


        curr_context = tuple(
            reversed(context)
        )


        if curr_context in boundary_context:

            context_count = sentence_count

        else:

            context_count = bigrams[curr_context]


        return math.log(
            adjusted_count / context_count
        )


    elif len(context) == 1:



        curr_gram = tuple(
            reversed([wi] + context)
        )


        c = bigrams.get(curr_gram, 0)




        if c == 0:

            log_alpha = alpha_bi[
                context[0]
            ]

            return (
                katz_probability(
                    wi,
                    []
                )
                + log_alpha
            )



        adjusted_count = (
            (c + 1)
            * Nc_bi(c + 1)
            / Nc_bi(c)
        )


        curr_context = tuple(
            reversed(context)
        )


        if curr_context in boundary_context:

            context_count = sentence_count

        else:

            context_count = unigrams[
                curr_context[0]
            ]


        return math.log(
            adjusted_count / context_count
        )

In [13]:
import time

alpha_bi = {}

vocab = set(unigrams.keys())

# Calculate total unigram GT probability once
total_unigram_prob = 0

for w in vocab:

    c = unigrams[w]

    if c == 0:
        adjusted_count = Nc_uni(1)
    else:
        adjusted_count = (c + 1) * (
            Nc_uni(c + 1) / Nc_uni(c)
        )

    total_unigram_prob += adjusted_count / n


# Group observed words by history
words_associated = defaultdict(set)

for h, w in bigrams:
    words_associated[h].add(w)


# Calculate alpha for every history
start_time = time.time()
total_histories = len(words_associated)

for i, h in enumerate(words_associated):



    if h == "<s>":
        context_count = sentence_count
    else:
        context_count = unigrams[h]

    seen_prob = 0
    seen_lower_prob = 0

    for w in words_associated[h]:

        c = bigrams[(h, w)]

        # GT discounted bigram probability
        adjusted_count = (c + 1) * (
            Nc_bi(c + 1) / Nc_bi(c)
        )

        seen_prob += adjusted_count / context_count

        # Lower-order unigram probability
        unigram_prob = math.exp(
            katz_probability(w, [])
        )

        seen_lower_prob += unigram_prob

    beta = 1 - seen_prob

    # Probability mass of words unseen after h
    unseen_prob = total_unigram_prob - seen_lower_prob

    alpha_bi[h] = math.log(beta / unseen_prob)


print(f"Done!")
print(f"Total time: {time.time() - start_time:.2f}s")
print(f"Alpha values calculated: {len(alpha_bi)}")

Done!
Total time: 70.74s
Alpha values calculated: 755567


In [14]:
import time
import math

# 1. GT discounted counts

D_bi = {
    c: (c + 1) * Nc_bi(c + 1) / Nc_bi(c)
    for c in freq_table_bi
}

D_tri = {
    c: (c + 1) * Nc_tri(c + 1) / Nc_tri(c)
    for c in freq_table_tri
}

D_uni = {
    c: (c + 1) * Nc_uni(c + 1) / Nc_uni(c)
    for c in freq_table_uni
}

print("GT discounts ready")



gt_uni_prob = {
    w: D_uni[c] / n
    for w, c in unigrams.items()
}

print("Unigram probabilities ready")


# 2. Sort trigram keys by history

start = time.time()

sorted_tri = sorted(trigrams.items())

print(
    f"Trigram sorting done in {time.time() - start:.2f}s"
)




exp_alpha_bi = {
    h: math.exp(a)
    for h, a in alpha_bi.items()
}



alpha_tri = {}

start_time = time.time()

current_history = None
seen_prob = 0.0
seen_lower_prob = 0.0
context_count = 0

history_count = 0


def finish_history(h):
    beta = 1.0 - seen_prob
    unseen_prob = 1.0 - seen_lower_prob

    alpha_tri[h] = math.log(beta / unseen_prob)


for i, ((h2, h1, w), c) in enumerate(sorted_tri):

    h = (h1, h2)

    # New history
    if h != current_history:

        # Finish previous history
        if current_history is not None:
            finish_history(current_history)

        current_history = h
        history_count += 1

        # C(h)
        if h == ("<s>", "<s>"):
            context_count = sentence_count
        else:
            context_count = bigrams[h]

        seen_prob = 0.0
        seen_lower_prob = 0.0



    # GT discounted trigram probability

    adjusted_tri = D_tri[c]

    seen_prob += adjusted_tri / context_count


    # Lower-order bigram

    c_bi = bigrams.get((h1, w), 0)

    if c_bi:

        adjusted_bi = D_bi[c_bi]

        if h1 == "<s>":
            lower_prob = adjusted_bi / sentence_count
        else:
            lower_prob = adjusted_bi / unigrams[h1]

    else:

        lower_prob = (
            exp_alpha_bi[h1]
            * gt_uni_prob[w]
        )

    seen_lower_prob += lower_prob


# Finish final history
if current_history is not None:
    finish_history(current_history)




elapsed = time.time() - start_time

print()
print(f"TRI alpha done in {elapsed:.2f}s")
print(f"Alpha values calculated: {len(alpha_tri):,}")

GT discounts ready
Unigram probabilities ready
Trigram sorting done in 215.62s


ZeroDivisionError: float division by zero

In [15]:
import time
import math


# PRECOMPUTE GT DISCOUNTED COUNTS

D_bi = {
    c: (c + 1) * Nc_bi(c + 1) / Nc_bi(c)
    for c in freq_table_bi
}

D_tri = {
    c: (c + 1) * Nc_tri(c + 1) / Nc_tri(c)
    for c in freq_table_tri
}

D_uni = {
    c: (c + 1) * Nc_uni(c + 1) / Nc_uni(c)
    for c in freq_table_uni
}

print("GT discounts ready")


# PRECOMPUTE GT UNIGRAM PROBABILITIES

gt_uni_prob = {
    w: D_uni[c] / n
    for w, c in unigrams.items()
}

print("Unigram probabilities ready")


exp_alpha_bi = {
    h: math.exp(a)
    for h, a in alpha_bi.items()
}

print("Bigram alpha values ready")


print(f"Trigrams available: {len(sorted_tri):,}")


# CALCULATE ALPHA_TRI

alpha_tri = {}

start_time = time.time()

current_history = None

seen_prob = 0.0
seen_lower_prob = 0.0
context_count = 0

history_count = 0

total_trigrams = len(sorted_tri)


for i, ((h2, h1, w), c) in enumerate(sorted_tri):

    # Katz history:
    #
    # h = (wi-1, wi-2)
    #
    h = (h1, h2)



    if h != current_history:

        # Finish previous history
        if current_history is not None:

            beta = 1.0 - seen_prob

            unseen_prob = 1.0 - seen_lower_prob

            if beta > 0 and unseen_prob > 0:
                alpha_tri[current_history] = math.log(
                    beta / unseen_prob
                )
            else:
                alpha_tri[current_history] = float("-inf")


        # Start new history
        current_history = h

        history_count += 1





        if h == ("<s>", "<s>"):

            context_count = sentence_count

        else:

            context_count = bigrams[(h[1], h[0])]


        seen_prob = 0.0
        seen_lower_prob = 0.0




    # GT-DISCOUNTED TRIGRAM PROBABILITY

    adjusted_tri = D_tri[c]

    seen_prob += adjusted_tri / context_count


    #
    # P_bo(wi | wi-1)

    c_bi = bigrams.get((h1, w), 0)


    if c_bi > 0:

        adjusted_bi = D_bi[c_bi]


        if h1 == "<s>":

            lower_prob = (
                adjusted_bi / sentence_count
            )

        else:

            lower_prob = (
                adjusted_bi / unigrams[h1]
            )


    else:

        # BACKOFF TO UNIGRAM
        #
        # alpha_bi(h1) * P_GT(w)

        lower_prob = (
            exp_alpha_bi[h1]
            * gt_uni_prob[w]
        )


    seen_lower_prob += lower_prob


# FINISH FINAL HISTORY

if current_history is not None:

    beta = 1.0 - seen_prob

    unseen_prob = 1.0 - seen_lower_prob

    if beta > 0 and unseen_prob > 0:

        alpha_tri[current_history] = math.log(
            beta / unseen_prob
        )

    else:

        alpha_tri[current_history] = float("-inf")



elapsed = time.time() - start_time

print()
print("=" * 60)
print(f"TRI alpha calculation completed")
print(f"Time: {elapsed:.2f}s")
print(f"Histories: {history_count:,}")
print(f"Alpha values: {len(alpha_tri):,}")
print("=" * 60)

GT discounts ready
Unigram probabilities ready
Bigram alpha values ready
Trigrams available: 10,834,281

TRI alpha calculation completed
Time: 277.63s
Histories: 6,000,634
Alpha values: 6,000,634


In [16]:
import time
import math

# 1. PRECOMPUTE GT DISCOUNTED COUNTS

D_bi = {
    c: (c + 1) * Nc_bi(c + 1) / Nc_bi(c)
    for c in freq_table_bi
}

D_tri = {
    c: (c + 1) * Nc_tri(c + 1) / Nc_tri(c)
    for c in freq_table_tri
}

D_quad = {
    c: (c + 1) * Nc_quad(c + 1) / Nc_quad(c)
    for c in freq_table_quad
}

D_uni = {
    c: (c + 1) * Nc_uni(c + 1) / Nc_uni(c)
    for c in freq_table_uni
}

print("GT discounts ready")


# 2. PRECOMPUTE GT UNIGRAM PROBABILITIES

gt_uni_prob = {
    w: D_uni[c] / n
    for w, c in unigrams.items()
}

print("Unigram probabilities ready")


# 3. PRECOMPUTE EXP(ALPHA)

exp_alpha_bi = {
    h: math.exp(a)
    for h, a in alpha_bi.items()
}

exp_alpha_tri = {
    h: math.exp(a)
    for h, a in alpha_tri.items()
}

print("Backoff factors ready")


# 4. SORT QUADGRAMS
#
# quadgrams are stored chronologically:
#
# (wi-3, wi-2, wi-1, wi)
#
# Katz history:
#
# (wi-1, wi-2, wi-3)

start_sort = time.time()

sorted_quad = sorted(quadgrams.items())

print(
    f"Quadgram sorting done in "
    f"{time.time() - start_sort:.2f}s"
)

print(f"Quadgrams: {len(sorted_quad):,}")


# 5. CALCULATE ALPHA_QUAD

alpha_quad = {}

start_time = time.time()

current_history = None

seen_prob = 0.0
seen_lower_prob = 0.0
context_count = 0

history_count = 0


def finish_quad_history(h):

    beta = 1.0 - seen_prob

    unseen_prob = 1.0 - seen_lower_prob

    if beta > 0 and unseen_prob > 0:

        alpha_quad[h] = math.log(
            beta / unseen_prob
        )

    else:

        alpha_quad[h] = float("-inf")


for i, ((h3, h2, h1, w), c) in enumerate(sorted_quad):

    # Katz history:
    #
    # h = (wi-1, wi-2, wi-3)

    h = (h1, h2, h3)


    # NEW HISTORY

    if h != current_history:

        # Finish previous history
        if current_history is not None:
            finish_quad_history(current_history)


        current_history = h
        history_count += 1



        if h == ("<s>", "<s>", "<s>"):

            context_count = sentence_count

        else:

            context_count = trigrams[
                (h[2], h[1], h[0])
            ]


        seen_prob = 0.0
        seen_lower_prob = 0.0






    adjusted_quad = D_quad[c]

    seen_prob += (
        adjusted_quad / context_count
    )


    tri_key = (h2, h1, w)

    c_tri = trigrams.get(tri_key, 0)


    if c_tri > 0:

        adjusted_tri = D_tri[c_tri]


        # C(wi-2, wi-1)
        if (h2, h1) == ("<s>", "<s>"):

            tri_context_count = sentence_count

        else:

            tri_context_count = bigrams[
                (h2, h1)
            ]


        lower_prob = (
            adjusted_tri / tri_context_count
        )


    else:
        tri_history = (h1, h2)

        c_bi = bigrams.get(
            (h1, w),
            0
        )


        if c_bi > 0:

            adjusted_bi = D_bi[c_bi]


            if h1 == "<s>":

                bi_prob = (
                    adjusted_bi / sentence_count
                )

            else:

                bi_prob = (
                    adjusted_bi / unigrams[h1]
                )


        else:

            bi_prob = (
                exp_alpha_bi[h1]
                * gt_uni_prob[w]
            )


        lower_prob = (
            exp_alpha_tri[tri_history]
            * bi_prob
        )


    seen_lower_prob += lower_prob


if current_history is not None:
    finish_quad_history(current_history)



elapsed = time.time() - start_time

print()
print("=" * 60)
print("QUAD alpha calculation completed")
print(f"Time: {elapsed:.2f}s")
print(f"Histories: {history_count:,}")
print(f"Alpha values: {len(alpha_quad):,}")
print("=" * 60)

GT discounts ready
Unigram probabilities ready
Backoff factors ready
Quadgram sorting done in 249.77s
Quadgrams: 13,060,259

QUAD alpha calculation completed
Time: 464.49s
Histories: 10,764,144
Alpha values: 10,764,144


### Testing Model

In [17]:
def katz_model_uni(sentence):
    words = sentence.split()
    return sum(
        katz_probability(wi, [])
        for wi in words
    )


def katz_model_bi(sentence):
    words = sentence.split()
    total_log_prob = 0

    for i, wi in enumerate(words):
        context = ["<s>"] if i == 0 else [words[i - 1]]
        total_log_prob += katz_probability(wi, context)

    return total_log_prob


def katz_model_tri(sentence):
    words = sentence.split()
    total_log_prob = 0

    for i, wi in enumerate(words):

        if i == 0:
            context = ["<s>", "<s>"]
        elif i == 1:
            context = [words[i - 1], "<s>"]
        else:
            context = [words[i - 1], words[i - 2]]

        total_log_prob += katz_probability(wi, context)

    return total_log_prob


def katz_model_quad(sentence):
    words = sentence.split()
    total_log_prob = 0

    for i, wi in enumerate(words):

        if i == 0:
            context = ["<s>", "<s>", "<s>"]
        elif i == 1:
            context = [words[i - 1], "<s>", "<s>"]
        elif i == 2:
            context = [words[i - 1], words[i - 2], "<s>"]
        else:
            context = [
                words[i - 1],
                words[i - 2],
                words[i - 3]
            ]

        total_log_prob += katz_probability(wi, context)

    return total_log_prob

In [18]:
katz_avg_log_prob_uni = 0
katz_avg_log_prob_bi = 0
katz_avg_log_prob_tri = 0
katz_avg_log_prob_quad = 0

total_tokens = 0

for sentence in test:

    total_tokens += len(sentence.split())

    katz_avg_log_prob_uni += katz_model_uni(sentence)
    katz_avg_log_prob_bi += katz_model_bi(sentence)
    katz_avg_log_prob_tri += katz_model_tri(sentence)
    katz_avg_log_prob_quad += katz_model_quad(sentence)


katz_avg_log_prob_uni /= total_tokens
katz_avg_log_prob_bi /= total_tokens
katz_avg_log_prob_tri /= total_tokens
katz_avg_log_prob_quad /= total_tokens


print(f"Average log_prob of Katz Unigram: {katz_avg_log_prob_uni}")
print(f"Average log_prob of Katz Bigram: {katz_avg_log_prob_bi}")
print(f"Average log_prob of Katz Trigram: {katz_avg_log_prob_tri}")
print(f"Average log_prob of Katz Quadgram: {katz_avg_log_prob_quad}")


pp_katz_uni = math.exp(-katz_avg_log_prob_uni)
pp_katz_bi = math.exp(-katz_avg_log_prob_bi)
pp_katz_tri = math.exp(-katz_avg_log_prob_tri)
pp_katz_quad = math.exp(-katz_avg_log_prob_quad)


print("Katz Unigram perplexity:", pp_katz_uni)
print("Katz Bigram perplexity:", pp_katz_bi)
print("Katz Trigram perplexity:", pp_katz_tri)
print("Katz Quadgram perplexity:", pp_katz_quad)

Average log_prob of Katz Unigram: -8.921881151676805
Average log_prob of Katz Bigram: -4.735829038240504
Average log_prob of Katz Trigram: -2.150209290388314
Average log_prob of Katz Quadgram: -1.278869875269004
Katz Unigram perplexity: 7494.173655401847
Katz Bigram perplexity: 113.95789501992736
Katz Trigram perplexity: 8.586655313557564
Katz Quadgram perplexity: 3.5925773699835535


### Stupid Backoff

In [19]:
def stupid_backoff(sentence):
    curr = ["<s>", "<s>", "<s>"] + sentence.split() + ["</s>"]
    ans =0
    for i in range(3,len(curr)):
        complete = [curr[i-3], curr[i-2], curr[i-1], curr[i]]
        if quadgrams.get(tuple(complete), 0) !=0:
            ans += math.log((quadgrams[tuple(complete) ]/trigrams[tuple(complete[1:])]))
        elif trigrams.get(tuple(complete[1:]), 0) !=0:
            ans += math.log(0.4*(trigrams[tuple(complete[1:])]/bigrams[tuple(complete[2:])]))
        elif bigrams.get(tuple(complete[2:]), 0) !=0:
            ans += math.log(0.4*( bigrams[tuple(complete[2:])]/ unigrams[complete[-1]]))
        else:
            ans += math.log(0.4 * (unigrams[complete[-1]]/n))
    return ans
        


In [20]:
sb_avg_log_prob = 0
total_tokens = 0

for sentence in test:
    total_tokens += len(sentence.split()) + 1
    sb_avg_log_prob += stupid_backoff(sentence)

sb_avg_log_prob /= total_tokens

print(f"Average log_prob of Stupid Backoff: {sb_avg_log_prob}")

pp_sb = math.exp(-sb_avg_log_prob)

print("Stupid Backoff perplexity:", pp_sb)

Average log_prob of Stupid Backoff: -0.7965471974122561
Stupid Backoff perplexity: 2.217869826037782


### Kneeser Ney Smoothing

In [21]:

D = 0.75
# Number of unique words that appear as the second word
# in a bigram
continuation_count = defaultdict(int)

# Number of unique contexts for each word
context_count = defaultdict(int)

for (w1, w2) in bigrams:
    continuation_count[w2] += 1
    context_count[w1] += 1


# Total number of unique bigram types
total_bigram_types = len(bigrams)

In [22]:
def kn_unigram_probability(w):
    return continuation_count.get(w, 0) / total_bigram_types

In [23]:
def kn_bigram_probability(wi, wi_1):

    context = (wi_1,)

    c_bigram = bigrams.get((wi_1, wi), 0)
    c_context = unigrams.get(wi_1, 0)

    if c_context == 0:
        return kn_unigram_probability(wi)

    lambda_value = (
        D * context_count.get(wi_1, 0)
        / c_context
    )

    first_term = max(c_bigram - D, 0) / c_context

    return (
        first_term
        + lambda_value * kn_unigram_probability(wi)
    )

In [24]:
# Number of distinct words that follow each bigram context
trigram_context_count = defaultdict(int)

for (w1, w2, w3) in trigrams:
    trigram_context_count[(w1, w2)] += 1

def kn_trigram_probability(wi, wi_1, wi_2):

    trigram = (wi_2, wi_1, wi)

    c_trigram = trigrams.get(trigram, 0)

    c_context = bigrams.get((wi_2, wi_1), 0)

    if c_context == 0:
        return kn_bigram_probability(wi, wi_1)

    lambda_value = (
        D
        * trigram_context_count.get((wi_2, wi_1), 0)
        / c_context
    )

    first_term = max(c_trigram - D, 0) / c_context

    return (
        first_term
        + lambda_value
        * kn_bigram_probability(wi, wi_1)
    )

In [25]:
quadgram_context_count = defaultdict(int)

for (w1, w2, w3, w4) in quadgrams:
    quadgram_context_count[(w1, w2, w3)] += 1

def kn_quadgram_probability(wi, wi_1, wi_2, wi_3):

    quadgram = (
        wi_3,
        wi_2,
        wi_1,
        wi
    )

    c_quadgram = quadgrams.get(quadgram, 0)

    c_context = trigrams.get(
        (wi_3, wi_2, wi_1),
        0
    )

    if c_context == 0:
        return kn_trigram_probability(
            wi,
            wi_1,
            wi_2
        )

    lambda_value = (
        D
        * quadgram_context_count.get(
            (wi_3, wi_2, wi_1),
            0
        )
        / c_context
    )

    first_term = (
        max(c_quadgram - D, 0)
        / c_context
    )

    return (
        first_term
        + lambda_value
        * kn_trigram_probability(
            wi,
            wi_1,
            wi_2
        )
    )

In [26]:
def kn_model_uni(sentence):

    words = sentence.split()

    ans = 0

    for wi in words:

        prob = kn_unigram_probability(wi)

        if prob == 0:
            return float("-inf")

        ans += math.log(prob)

    return ans

def kn_model_bi(sentence):

    words = sentence.split()

    ans = 0

    for i, wi in enumerate(words):

        if i == 0:
            prob = kn_bigram_probability(wi, "<s>")
        else:
            prob = kn_bigram_probability(
                wi,
                words[i - 1]
            )

        if prob == 0:
            return float("-inf")

        ans += math.log(prob)

    return ans

def kn_model_tri(sentence):

    words = sentence.split()

    ans = 0

    for i, wi in enumerate(words):

        if i == 0:

            prob = kn_trigram_probability(
                wi,
                "<s>",
                "<s>"
            )

        elif i == 1:

            prob = kn_trigram_probability(
                wi,
                words[i - 1],
                "<s>"
            )

        else:

            prob = kn_trigram_probability(
                wi,
                words[i - 1],
                words[i - 2]
            )

        if prob == 0:
            return float("-inf")

        ans += math.log(prob)

    return ans

def kn_model_quad(sentence):

    words = sentence.split()

    ans = 0

    for i, wi in enumerate(words):

        if i == 0:

            prob = kn_quadgram_probability(
                wi,
                "<s>",
                "<s>",
                "<s>"
            )

        elif i == 1:

            prob = kn_quadgram_probability(
                wi,
                words[i - 1],
                "<s>",
                "<s>"
            )

        elif i == 2:

            prob = kn_quadgram_probability(
                wi,
                words[i - 1],
                words[i - 2],
                "<s>"
            )

        else:

            prob = kn_quadgram_probability(
                wi,
                words[i - 1],
                words[i - 2],
                words[i - 3]
            )

        if prob == 0:
            return float("-inf")

        ans += math.log(prob)

    return ans

In [27]:
kn_avg_log_prob_uni = 0
kn_avg_log_prob_bi = 0
kn_avg_log_prob_tri = 0
kn_avg_log_prob_quad = 0

total_tokens = 0

for sentence in test:

    total_tokens += len(sentence.split())

    kn_avg_log_prob_uni += kn_model_uni(sentence)
    kn_avg_log_prob_bi += kn_model_bi(sentence)
    kn_avg_log_prob_tri += kn_model_tri(sentence)
    kn_avg_log_prob_quad += kn_model_quad(sentence)


kn_avg_log_prob_uni /= total_tokens
kn_avg_log_prob_bi /= total_tokens
kn_avg_log_prob_tri /= total_tokens
kn_avg_log_prob_quad /= total_tokens


print("Average log_prob of KN Unigram:",
      kn_avg_log_prob_uni)

print("Average log_prob of KN Bigram:",
      kn_avg_log_prob_bi)

print("Average log_prob of KN Trigram:",
      kn_avg_log_prob_tri)

print("Average log_prob of KN Quadgram:",
      kn_avg_log_prob_quad)


pp_kn_uni = math.exp(-kn_avg_log_prob_uni)
pp_kn_bi = math.exp(-kn_avg_log_prob_bi)
pp_kn_tri = math.exp(-kn_avg_log_prob_tri)
pp_kn_quad = math.exp(-kn_avg_log_prob_quad)


print("KN Unigram perplexity:", pp_kn_uni)
print("KN Bigram perplexity:", pp_kn_bi)
print("KN Trigram perplexity:", pp_kn_tri)
print("KN Quadgram perplexity:", pp_kn_quad)

Average log_prob of KN Unigram: -9.258774689461614
Average log_prob of KN Bigram: -5.201622889522005
Average log_prob of KN Trigram: -2.9001998082458367
Average log_prob of KN Quadgram: -1.9328253387952163
KN Unigram perplexity: 10496.264274530171
KN Bigram perplexity: 181.56666554104154
KN Trigram perplexity: 18.177777076359323
KN Quadgram perplexity: 6.909002965955287
